# reranker_colab.ipynb — 리랭커(BAAI/bge-reranker-v2-m3)를 코랩 런타임에서 돌리기 위한 노트북

배경: 로컬 컴퓨터(RAM 7.4GB)에서는 bge-reranker-v2-m3(568M 파라미터)를 로드하려고 하면
세그멘테이션 폴트로 프로세스가 죽는다(2026-08-13 실측 확인) — 임베딩(e5-small/e5-large)은
디스크 공간을 확보한 뒤 정상 동작했지만, 리랭커는 모델 자체가 더 커서 로컬 RAM으로는
부족한 것으로 보인다.

이 노트북은 VSCode의 Colab 확장으로 이 파일에 연결해서, 무거운 모델(리랭커·임베딩)만
코랩의 더 넉넉한 런타임에서 실행하기 위한 것이다. `agent/` 패키지 코드는 건드리지 않는다 —
같은 코드를 어디서 실행하느냐만 다르게 하는 것이 목표(로컬 코드와 갈라지지 않게).

## 사용법
1. VSCode에서 이 노트북 열기 → 우측 상단 "커널 선택" → "Colab" → "New Colab Server" → 구글 계정 로그인
2. 아래 셀을 순서대로 실행해서 (1) 연결 확인 → (2) 리랭커 로드 확인까지 검증
3. 여기까지 되면, 다음 단계로 실제 배치 파이프라인과 연동하는 방법을 이어서 설계한다

## 1. 연결 확인 — 이게 코랩에서 도는지, 로컬에서 도는지부터 확인

In [ ]:
import platform, os
print("플랫폼:", platform.platform())
print("코랩 여부(google.colab 모듈 있는지):", end=" ")
try:
    import google.colab  # noqa: F401
    print("코랩에서 실행 중")
except ImportError:
    print("코랩 아님 — 로컬에서 도는 중 (커널 선택이 안 된 상태일 수 있음)")

# 메모리 확인 (코랩이면 보통 12GB+, Pro면 더 많음)
!cat /proc/meminfo 2>/dev/null | head -3 || echo "(Linux 환경 아님 — Windows 로컬일 가능성)"

## 2. 리랭커(bge-reranker-v2-m3) 로드 테스트
로컬에서 세그멘테이션 폴트로 죽었던 바로 그 모델. 여기서 정상 로드되는지 확인.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import time
from sentence_transformers import CrossEncoder

t0 = time.time()
model = CrossEncoder("BAAI/bge-reranker-v2-m3")
print(f"로드 성공! 소요: {time.time() - t0:.1f}초")

score = model.predict([("소매 판매가 늘었다", "소매판매액지수")])
print("테스트 점수:", score)

## 3. 실제 파이프라인 연동

로컬에서 `python -m agent.pipeline.export_for_rerank --csv --csv-n 30`을 실행하면
`data/rerank_pending.json`이 생긴다 — 1~3단계(키워드매칭+임베딩+VDB 병합까지 전부 로컬에서
끝낸 claim별 최종 후보 목록, `merged_candidates`)가 들어있다.

(2026-08-15: 예전엔 임베딩 매칭도 이 노트북에서 했는데, 여러 claim을 한 번의 encode()
호출로 배치 처리하면 로컬에서도 세그폴트 없이 안전하다는 게 확인돼서, 임베딩(64개
카탈로그 비교)과 VDB(KOSIS 표 28만7천여 개, agent/kosis/chroma_db) 조회를 전부 로컬로
옮겼다. 이제 코랩은 로컬에서 못 돌리는 리랭커(bge-reranker-v2-m3, 로드 자체가 세그폴트)
만 담당한다 — 그래서 이 노트북엔 리랭킹 셀만 남아있다.)

이 파일을 아래 셀에서 업로드하면, 여기서 리랭커로 점수를 매기고 `rerank_results.json`을
다시 다운로드해준다. 그걸 로컬 `data/` 폴더에 넣고
`python -m agent.pipeline.resume_after_rerank`를 실행하면 4~8단계가 마저 진행된다.

점수 계산 방식(시그모이드 → 순위 기반 verified 승격)은 `agent/mapping/reranker.py`의
`rerank()`/`_promote_verified_within_top_ranks()`랑 똑같이 맞춰뒀다 — 로컬에서 리랭커가
직접 돌 때랑 결과가 갈라지지 않게.

In [ ]:
import shutil
from google.colab import drive

drive.mount("/content/drive")

# 아래 SRC 경로를 본인 드라이브에 업로드한 rerank_pending.json 실제 경로로 바꾸세요.
# (구글 드라이브 웹사이트나 드라이브 앱에서 이 파일을 "내 드라이브" 최상위에 끌어다 놓으면
# 기본값 그대로 써도 됩니다.)
SRC = "/content/drive/MyDrive/rerank_pending.json"
pending_filename = "rerank_pending.json"
shutil.copy(SRC, pending_filename)
print("복사됨:", pending_filename)

In [ ]:
import json

with open(pending_filename, encoding="utf-8") as f:
    pending = json.load(f)

# 2026-08-16: 아래 리랭킹 셀의 doc_text_for()가 bare `catalog`를 참조하는데, 이 셀이
# pending["catalog"]만 있고 catalog라는 이름 자체를 assign한 적이 없어서
# NameError("catalog")가 났다(실측 확인). 여기서 명시적으로 꺼내둔다.
catalog = pending["catalog"]

print(f"리랭킹 대상 claim {len(pending['items'])}건, 카탈로그 {len(catalog)}개 표")

In [ ]:
import math


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))


# agent/mapping/reranker.py의 _promote_verified_within_top_ranks와 반드시 같은 로직으로
# 유지 — 로컬 경로랑 코랩 경로의 판정 방식이 갈라지면 안 됨.
#
# 2026-08-14: 기존엔 keyword로 검증된(verified) 후보에 고정 보너스(+0.05)를 점수에
# 더했는데, 리랭커 raw score가 거의 항상 0 근처라(시그모이드 통과 후 후보 간 점수
# 스프레드가 0.003 정도밖에 안 됨) +0.05가 꼴찌 후보도 1등으로 만들어버릴 수 있었다.
# 실제로 keyword_search 오탐("유가"가 "유가증권"과 혼동)으로 무관한 표가 verified
# 보너스 덕에, 리랭커가 이미 훨씬 높게 평가해둔 진짜 정답(unverified)을 이기는 사례가
# 확인됐다(이 표의 raw_rerank_score를 직접 봐도 최하위였다). 그래서 점수에 더하는 대신,
# verified 후보가 순수 리랭킹 순위 상위 VERIFIED_PROMOTION_RANK 안에 들 때만 1등으로
# 승격시키는 방식(RRF류 순위 기반 판단)으로 바꿨다 — 리랭커가 확실히 아니라고 판단한
# 경우(순위가 한참 밀림)까지 verified라는 이유만으로 뒤집지 않는다.
VERIFIED_PROMOTION_RANK = 3


def is_verified(c):
    return "unverified" not in (c.get("source_meta") or "")


def promote_verified_within_top_ranks(ranked):
    if not ranked or is_verified(ranked[0]):
        return ranked
    for i, c in enumerate(ranked[:VERIFIED_PROMOTION_RANK]):
        if is_verified(c):
            return [c] + ranked[:i] + ranked[i + 1 :]
    return ranked


def doc_text_for(c):
    # 2026-08-15: merged_candidates엔 이제 64개 카탈로그 후보뿐 아니라 VDB(KOSIS 표
    # 28만7천여 개) 후보도 섞여 있는데, pending["catalog"]엔 64개 카탈로그 정보만 있어서
    # VDB 표 ID로 조회하면 KeyError가 났다(실측 확인). VDB 후보는 자기 자신의 table_name을
    # 이미 candidate dict에 들고 있으니 그걸 폴백으로 쓴다.
    entry = catalog.get(c["table_id"])
    if entry is not None:
        return entry["embedding_text"]
    return c.get("table_name") or c["table_id"]


output_items = []
for i, item in enumerate(pending["items"]):
    claim_sentence = item["claim"]["sentence"]
    candidates = item["merged_candidates"]
    if not candidates:
        output_items.append({"item_id": item["item_id"], "candidates": []})
        continue
    docs = [doc_text_for(c) for c in candidates]

    raw_scores = model.predict([(claim_sentence, d) for d in docs])

    reranked = []
    for c, raw in zip(candidates, raw_scores):
        reranked.append({**c, "score": sigmoid(float(raw)), "raw_rerank_score": float(raw)})
    reranked.sort(key=lambda c: c["score"], reverse=True)
    reranked = promote_verified_within_top_ranks(reranked)

    output_items.append({"item_id": item["item_id"], "candidates": reranked[:5]})
    if i % 10 == 0:
        print(f"리랭킹 진행: {i}/{len(pending['items'])}")

print("리랭킹 완료:", len(output_items), "건")

In [ ]:
import shutil

with open("rerank_results.json", "w", encoding="utf-8") as f:
    json.dump({"items": output_items}, f, ensure_ascii=False, indent=2)

# files.download()는 이 VS Code<->코랩 연결에서는 성공 메시지만 찍히고 실제 브라우저
# 다운로드가 안 뜨는 문제가 있음(업로드 위젯 때와 동일한 제약, 2026-08-14 확인).
# 이미 마운트된 드라이브에 저장해서 드라이브 웹/앱에서 직접 받는 방식으로 우회.
DRIVE_OUT = "/content/drive/MyDrive/rerank_results.json"
shutil.copy("rerank_results.json", DRIVE_OUT)
print(f"저장 완료: {DRIVE_OUT}")
print("구글 드라이브(내 드라이브 최상위)에서 rerank_results.json을 다운로드해서")
print("로컬 data/ 폴더에 옮기고 resume_after_rerank.py를 실행하세요.")